# Streaming（流式请求 + 逐 Chunk 解析）

## 目标

本示例演示如何开启 Chat Completions 流式响应，逐个解析 `ChatCompletionChunk`，并把多个 `delta.content` 重建为完整回答。

## 前置条件

- Python 3.10+
- OpenAI Python SDK `openai>=1.0.0`，环境变量加载库 `python-dotenv>=1.0.0`
- 环境变量：`HY3_BASE_URL`、`HY3_API_KEY`、`HY3_MODEL`；可从 `quickstart/.env.example` 复制为 `quickstart/.env`
- 模型能力要求：部署服务支持 Chat Completions 流式输出；如不支持 `stream_options.include_usage`，可删除该参数，正文流不受影响

安装依赖：

```bash
python -m pip install "openai>=1.0.0" "python-dotenv>=1.0.0"
```

## 完整请求

```python
stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "请用三句话说明流式响应适合什么场景。",
        }
    ],
    temperature=0.9,
    top_p=1.0,
    max_tokens=128,
    stream=True,
    stream_options={"include_usage": True},
    extra_body={
        "chat_template_kwargs": {"reasoning_effort": "no_think"}
    },
)
```

## 完整 Response 解析

流式响应不是一个完整 message，而是一系列 chunk。首个 chunk 可能只包含 `role`，中间 chunk 才包含正文，最后还可能有 `finish_reason` 或只有 `usage`、没有 `choices`，因此不能固定读取 `chunk.choices[0]`。

```python
text_by_choice = {}
finish_reasons = {}
usage = None

for chunk in stream:
    usage = getattr(chunk, "usage", None) or usage
    for choice in chunk.choices:
        text_by_choice.setdefault(choice.index, [])
        content = getattr(choice.delta, "content", None)
        if content:
            text_by_choice[choice.index].append(content)
        if choice.finish_reason:
            finish_reasons[choice.index] = choice.finish_reason

full_text = "".join(text_by_choice[0])
print(full_text, finish_reasons.get(0), usage)
```

脚本还会逐 chunk 打印 `role`、`content`、`reasoning_content` 和 `finish_reason`，便于观察协议实际返回内容。

## 运行方式

在 `quickstart/` 目录执行：

```bash
python examples/02_streaming/streaming.py
```

## 示例输出

```text
chunk=1, choice=0, role='assistant', content=None, finish_reason=None
chunk=2, choice=0, role=None, content='流式', finish_reason=None
chunk=3, choice=0, role=None, content='响应适合需要快速反馈的交互场景。', finish_reason=None
chunk=4, choice=0, role=None, content=None, finish_reason='stop'

=== 重建后的完整响应 ===
id=chatcmpl-example
model=hy3
choice[0].finish_reason=stop
choice[0].content=流式响应适合需要快速反馈的交互场景。它能降低用户感知等待时间。客户端需逐块拼接输出。
usage: prompt=22, completion=38, total=60
```

Chunk 边界、ID、正文和 Token 数均为示意，实际结果由服务端决定。


In [ ]:
"""Hy3 流式请求与逐 chunk 解析示例。"""

import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI


def load_project_env() -> None:
    candidates = [Path.cwd() / ".env", Path.cwd() / "quickstart" / ".env"]
    if "__file__" in globals():
        candidates.insert(0, Path(__file__).resolve().parents[2] / ".env")
    for candidate in candidates:
        if candidate.is_file():
            load_dotenv(candidate, override=False)
            return


load_project_env()

BASE_URL = os.getenv("HY3_BASE_URL", "http://127.0.0.1:8000/v1")
API_KEY = os.getenv("HY3_API_KEY", "EMPTY")
MODEL = os.getenv("HY3_MODEL", "hy3")
MAX_TOKENS = int(os.getenv("HY3_MAX_TOKENS", "128"))
REASONING_EFFORT = os.getenv("HY3_REASONING_EFFORT", "no_think")


def main() -> None:
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "请用三句话说明流式响应适合什么场景。",
            }
        ],
        temperature=0.9,
        top_p=1.0,
        max_tokens=MAX_TOKENS,
        stream=True,
        stream_options={"include_usage": True},
        extra_body={
            "chat_template_kwargs": {
                "reasoning_effort": REASONING_EFFORT,
            }
        },
    )

    text_by_choice: dict[int, list[str]] = {}
    finish_reasons: dict[int, str] = {}
    response_id = None
    response_model = None
    usage = None

    for chunk_number, chunk in enumerate(stream, start=1):
        response_id = response_id or chunk.id
        response_model = response_model or chunk.model
        usage = getattr(chunk, "usage", None) or usage

        # usage chunk 可能没有 choices，因此必须遍历而不是固定访问 choices[0]。
        for choice in chunk.choices:
            index = choice.index
            delta = choice.delta
            text_by_choice.setdefault(index, [])

            role = getattr(delta, "role", None)
            content = getattr(delta, "content", None)
            reasoning = getattr(delta, "reasoning_content", None)

            print(
                f"chunk={chunk_number}, choice={index}, "
                f"role={role!r}, content={content!r}, "
                f"finish_reason={choice.finish_reason!r}"
            )
            if reasoning:
                print(f"  reasoning_content={reasoning!r}")
            if content:
                text_by_choice[index].append(content)
            if choice.finish_reason:
                finish_reasons[index] = choice.finish_reason

    print("\n=== 重建后的完整响应 ===")
    print(f"id={response_id}")
    print(f"model={response_model}")
    for index in sorted(text_by_choice):
        print(f"choice[{index}].finish_reason={finish_reasons.get(index)}")
        print(f"choice[{index}].content={''.join(text_by_choice[index])}")
    if usage:
        print(
            "usage: "
            f"prompt={usage.prompt_tokens}, "
            f"completion={usage.completion_tokens}, "
            f"total={usage.total_tokens}"
        )
    else:
        print("usage=当前服务未在流式响应中返回 usage")


if __name__ == "__main__":
    main()
